# PneumoScan AI — Training (Colab / Kaggle GPU)

Trains **DenseNet121** and **Swin-Tiny** for Stage 1 (Normal vs Pneumonia) and Stage 2 (Bacterial vs Viral)
with identical settings, then calibrates them. Outputs go to `backend/weights/` and `backend/metrics/metrics.json`.

**Runtime → Change runtime type → GPU** before starting.

> Data use: the Kermany dataset is public (CC BY 4.0). Any testing with real patient images requires consent and
> appropriate institutional approval.

## 1. Get the code
Upload/clone the project repository (replace the URL with yours), or upload a zip and unzip it.

In [ ]:
import os
REPO_URL = "https://github.com/<your-org>/pneumoscan-ai.git"  # <- change me
if not os.path.exists("pneumoscan-ai"):
    !git clone -q $REPO_URL pneumoscan-ai
%cd pneumoscan-ai/training
!pip -q install -r requirements.txt

## 2. Download the Kermany dataset
**Colab:** upload your `kaggle.json` API token (Kaggle → Account → Create New Token).
**Kaggle notebooks:** add the dataset *paultimothymooney/chest-xray-pneumonia* as input instead and set `DATA_ROOT`
to `/kaggle/input/chest-xray-pneumonia/chest_xray`.

In [ ]:
DATA_ROOT = "/content/data/chest_xray"
if not os.path.exists(DATA_ROOT):
    from google.colab import files  # noqa
    files.upload()  # select kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -q -d paultimothymooney/chest-xray-pneumonia -p /content/data
    !unzip -q -o /content/data/chest-xray-pneumonia.zip -d /content/data

## 3. Patient-grouped 70/15/15 split
Re-splits all images (the 16-image Kaggle `val` folder is not used) and removes duplicate files.

In [ ]:
!python prepare_data.py --data-root $DATA_ROOT --out data/splits.csv

## 4. Place trained weights
If you trained in another session, upload `pneumoscan_weights.zip` and unzip it at the repo root.

In [ ]:
# from google.colab import files; files.upload()
# !cd .. && unzip -o /content/pneumoscan_weights.zip
!ls ../backend/weights

## 5. Internal test-set evaluation
Metrics → `backend/metrics/metrics.json` (dashboard); figures → `training/outputs/figures/`.

In [ ]:
!python evaluate.py

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("outputs/figures/*.png"))[:12]:
    print(f); display(Image(f, width=320))

## 6. Grad-CAM gallery and lung-attention analysis

In [ ]:
for arch in ["densenet", "swin"]:
    !python gradcam_analysis.py --arch $arch --per-category 6 --max-images 400

## 7. External validation (Stage 1) — RSNA Pneumonia Detection
Requires accepting the competition rules on Kaggle. Uses a stratified subset to keep runtime reasonable.

In [ ]:
!kaggle competitions download -q -c rsna-pneumonia-detection-challenge -p /content/rsna
!unzip -q -o /content/rsna/rsna-pneumonia-detection-challenge.zip -d /content/rsna
!python external_validate.py --name "RSNA Pneumonia Detection" \
    --description "Adult CXRs (NIH ChestX-ray14 subset); Stage 1 only — no viral/bacterial labels." \
    --rsna-labels /content/rsna/stage_2_train_labels.csv --rsna-images /content/rsna/stage_2_train_images --limit 3000

## 8. Download results
Copy `backend/metrics/` (metrics.json + gallery) into your local project; the dashboard updates automatically.

In [ ]:
!cd .. && zip -qr /content/pneumoscan_metrics.zip backend/metrics training/outputs
try:
    from google.colab import files
    files.download("/content/pneumoscan_metrics.zip")
except ImportError:
    print("Results zipped at /content/pneumoscan_metrics.zip")